# 00 · Pipeline de Modelado — Etapa 4
### Detección de Incendios Forestales con Imágenes Satelitales
**Curso:** TTCT0017 — Computación Paralela y Distribuida, LEAD University

Este notebook corre **todo el flujo de la Etapa 4 de punta a punta** sobre `dataset_modelo.parquet`:

1. Explorar el dataset (registros, columnas, distribución de clases, nulos, correlaciones)
2. Preparar los datos (selección de variables, train/val/test, normalización)
3. Entrenar 3 modelos: 🌳 Random Forest, ⚡ XGBoost, 🧠 Red neuronal (PyTorch)
4. Comparar: Accuracy, Precision, Recall, F1, ROC-AUC, matriz de confusión
5. Generar automáticamente tablas, gráficas y figuras para el informe IEEE

> Nota: el archivo real (~1.83 GB) no se subió a este entorno de desarrollo, así que el
> notebook **auto-detecta el esquema** al cargar: usa las columnas conocidas de
> `preparar_dataset_modelo.py` si existen, y si algo cambió te avisa explícitamente en vez
> de fallar en silencio. Corre esta celda tal cual en Kabré/Colab con el archivo real.


## 0. Configuración

In [ ]:
#pip install xgboost

In [ ]:
#pip install plotly pandas polars matplotlib scikit-learn pyarrow  

In [ ]:
#pip install --user torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- Rutas (ABSOLUTAS, para que no importe desde donde arranque la sesion de Jupyter) ---
# Datos pesados (dataset, splits, modelos entrenados) -> viven en /data, NO en /home
USUARIO = os.path.basename(os.path.expanduser("~"))
DATOS_BASE_DIR = os.path.join("/data", USUARIO, "deteccion_incendios")
# Resultados livianos (tablas/figuras que van al repo de Git) -> se quedan en /home
BASE_DIR = os.path.join(os.path.expanduser("~"), "deteccion_incendios", "modelado")
RUTA_DATASET = os.path.join(DATOS_BASE_DIR, "data", "dataset_modelo.parquet")

# --- MODO DE EJECUCION --------------------------------------------------
# "dev"  -> muestra pequena, corre en segundos/minutos en tu laptop.
# "full" -> dataset completo, correr en Kabre (GPU + RAM suficiente).
MODO = "full"
FRACCION_DEV = 0.3
DIR_FIGURAS  = os.path.join(BASE_DIR, "resultados/figuras")   # PNG 150dpi listos para \includegraphics en IEEE
DIR_TABLAS   = os.path.join(BASE_DIR, "resultados/tablas")
DIR_MODELOS  = os.path.join(DATOS_BASE_DIR, "modelos")
os.makedirs(DIR_FIGURAS, exist_ok=True)
os.makedirs(DIR_TABLAS, exist_ok=True)
os.makedirs(DIR_MODELOS, exist_ok=True)

SEMILLA = 42
TARGET_ESPERADO = "es_falsa_alarma"

# Si el dataset no cabe comodo en memoria/GPU, se puede submuestrear SOLO para el
# entrenamiento de la red neuronal (RF y XGBoost trabajan bien con todo el dataset).
SUBSAMPLE_FRAC_NN = 1.0   # ej. 0.3 para usar 30% de los datos en la red neuronal

assert os.path.exists(RUTA_DATASET), (
    f"No se encontro {RUTA_DATASET}. Ajusta RUTA_DATASET a donde este tu dataset_modelo.parquet."
)
print("Dataset encontrado:", RUTA_DATASET, f"({os.path.getsize(RUTA_DATASET)/1e9:.2f} GB)")


In [ ]:
# Auto-deteccion de esquema: usamos scan_parquet (lazy) para no cargar el archivo completo aun
lf = pl.scan_parquet(RUTA_DATASET)
esquema = lf.collect_schema()
columnas = list(esquema.names())
print(f"Columnas detectadas ({len(columnas)}):", columnas)

COLUMNAS_ESPERADAS = ["delta_t", "brightness", "bright_t31", "frp", "scan", "track",
                       "mes", "hora", "es_noche", "latitude", "longitude", "es_falsa_alarma"]
faltantes = [c for c in COLUMNAS_ESPERADAS if c not in columnas]
extra = [c for c in columnas if c not in COLUMNAS_ESPERADAS]
if faltantes:
    print(f"AVISO: no estan en el archivo estas columnas esperadas: {faltantes}")
if extra:
    print(f"Columnas adicionales encontradas (no en el esquema base): {extra}")

TARGET = TARGET_ESPERADO if TARGET_ESPERADO in columnas else columnas[-1]
print(f"\nColumna objetivo (target): '{TARGET}'")


---
## 1. Explorar el dataset

### 1.1 Número de registros y columnas

In [ ]:
t0 = time.perf_counter()
n_filas_total = lf.select(pl.len()).collect().item()
n_cols = len(columnas)
print(f"Registros totales : {n_filas_total:,}")
print(f"Columnas          : {n_cols}")
print(f"(consulta en {time.perf_counter()-t0:.1f}s, sin materializar el archivo completo)")

# Aplicar submuestreo si estamos en modo dev (clave para que corra rapido en laptop)
if MODO == "dev":
    paso = max(1, int(1 / FRACCION_DEV))
    lf = lf.gather_every(paso)
    n_filas = lf.select(pl.len()).collect().item()
    print(f"\nModo dev: usando 1 de cada {paso} filas -> {n_filas:,} filas de trabajo")
else:
    n_filas = n_filas_total


### 1.2 Tipos de dato y valores faltantes

Se calcula con agregaciones perezosas (`streaming`) para que funcione aunque el archivo no quepa en RAM.

In [ ]:
nulos = lf.select([pl.col(c).null_count().alias(c) for c in columnas]).collect(engine="streaming")
resumen_nulos = pd.DataFrame({
    "columna": columnas,
    "tipo": [str(esquema[c]) for c in columnas],
    "nulos": [nulos[c][0] for c in columnas],
})
# .astype("float64") por seguridad, mismo tipo de riesgo de overflow que en balance
resumen_nulos["pct_nulos"] = 100 * resumen_nulos["nulos"].astype("float64") / n_filas
resumen_nulos = resumen_nulos.sort_values("pct_nulos", ascending=False)
resumen_nulos.to_csv(os.path.join(DIR_TABLAS, "01_resumen_nulos.csv"), index=False)
resumen_nulos


### 1.3 Distribución de la variable objetivo

In [ ]:
balance = (
    lf.group_by(TARGET).agg(pl.len().alias("n"))
    .sort(TARGET).collect(engine="streaming").to_pandas()
)
# .astype("float64") evita overflow de uint32 en datasets grandes (bug encontrado: 100*n se desborda)
balance["pct"] = 100 * balance["n"].astype("float64") / balance["n"].sum()
balance.to_csv(os.path.join(DIR_TABLAS, "02_balance_clases.csv"), index=False)
print(balance)

fig = px.pie(balance, names=TARGET, values="n", title=f"Distribucion de clases: {TARGET}")
fig.show()

# version matplotlib para el informe IEEE
plt.figure(figsize=(4,4))
plt.pie(balance["n"], labels=balance[TARGET], autopct="%1.1f%%", colors=["#2980b9","#c0392b"])
plt.title(f"Distribucion de clases: {TARGET}")
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "01_balance_clases.png"), dpi=150)
plt.close()


### 1.4 Correlaciones entre variables numéricas

Se usa una muestra (no todo el dataset) para calcular la matriz de correlación de forma económica en memoria.

In [ ]:
cols_numericas = [c for c, t in esquema.items() if t.is_numeric() and c != TARGET]
paso = max(1, n_filas // 500_000)  # muestra de ~500k filas
muestra = lf.select(cols_numericas + [TARGET]).gather_every(paso).collect(engine="streaming").to_pandas()

corr = muestra[cols_numericas].corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                 title="Matriz de correlacion (muestra)")
fig.show()

plt.figure(figsize=(7,6))
im = plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(cols_numericas)), cols_numericas, rotation=45, ha="right")
plt.yticks(range(len(cols_numericas)), cols_numericas)
plt.colorbar(im); plt.title("Matriz de correlacion")
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "02_correlaciones.png"), dpi=150)
plt.close()


---
## 2. Preparar los datos

### 2.1 Selección de variables

Se excluyen columnas ID/derivadas que no aportan al modelo, y el propio target.

In [ ]:
COLUMNAS_EXCLUIR = {TARGET}  # agrega aqui cualquier columna que no deba usarse como predictora
predictoras = [c for c in cols_numericas if c not in COLUMNAS_EXCLUIR]
print(f"Predictoras seleccionadas ({len(predictoras)}):", predictoras)


### 2.2 Split train / val / test (70/15/15, estratificado)

In [ ]:
from sklearn.model_selection import train_test_split

# Para datasets muy grandes, cargar directo a pandas puede ser costoso.
# Si tu maquina no tiene RAM suficiente para los ~50M de registros, ajusta aqui un
# muestreo previo (ej. .gather_every(2) para tomar el 50%) antes de pasar a pandas.
# cast a float32 para usar la mitad de RAM que float64, sin perdida relevante de precision
df = (
    lf.select(predictoras + [TARGET])
    .with_columns([pl.col(c).cast(pl.Float32) for c in predictoras])
    .collect(engine="streaming")
    .to_pandas()
)
print("Dataset cargado en memoria:", df.shape, f"(~{df.memory_usage(deep=True).sum()/1e6:.0f} MB)")

X = df[predictoras]
y = df[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEMILLA)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEMILLA)

for nombre, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{nombre:<6} n={len(yy):>12,}  pct_positivos={100*yy.mean():.2f}%")


### 2.3 Normalización

RF y XGBoost no la necesitan (son invariantes a escala), pero la red neuronal sí:
se estandariza (`StandardScaler`) ajustado **solo con train** para evitar fuga de datos.

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_val_esc   = escalador.transform(X_val)
X_test_esc  = escalador.transform(X_test)

joblib.dump(escalador, os.path.join(DIR_MODELOS, "escalador.pkl"))
print("Escalador ajustado y guardado.")


---
## 3. Entrenar los 3 modelos

## 2.4 Monitor de recursos (RAM y GPU)

Para que Robson tenga los datos de rendimiento que necesita (uso de memoria y GPU durante
el entrenamiento, no solo tiempos), esta celda define una utilidad que muestrea RAM (via
`psutil`) y memoria/uso de GPU (via `nvidia-smi`) en segundo plano mientras entrena cada
modelo, sin afectar el tiempo de entrenamiento medido.


In [ ]:
import threading, subprocess, os
try:
    import psutil
    HAY_PSUTIL = True
except ImportError:
    HAY_PSUTIL = False
    print("Aviso: psutil no esta instalado, RAM no se podra medir (pip install --user psutil)")

class MonitorRecursos:
    """Muestrea RAM y GPU en un hilo aparte mientras el bloque 'with' esta activo."""
    def __init__(self, intervalo=0.5):
        self.intervalo = intervalo
        self._corriendo = False
        self._hilo = None
        self.muestras_ram_mb = []
        self.muestras_gpu_mem_mb = []
        self.muestras_gpu_util_pct = []

    def _leer_gpu(self):
        try:
            salida = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
                 "--format=csv,noheader,nounits"], timeout=2
            ).decode().strip().split("\n")[0].split(",")
            return float(salida[0].strip()), float(salida[1].strip())
        except Exception:
            return None, None

    def _loop(self):
        proceso = psutil.Process(os.getpid()) if HAY_PSUTIL else None
        while self._corriendo:
            if proceso is not None:
                self.muestras_ram_mb.append(proceso.memory_info().rss / 1e6)
            gpu_mem, gpu_util = self._leer_gpu()
            if gpu_mem is not None:
                self.muestras_gpu_mem_mb.append(gpu_mem)
                self.muestras_gpu_util_pct.append(gpu_util)
            threading.Event().wait(self.intervalo)

    def __enter__(self):
        self._corriendo = True
        self._hilo = threading.Thread(target=self._loop, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *exc):
        self._corriendo = False
        self._hilo.join(timeout=2)

    def resumen(self):
        def _pico_prom(lista):
            return (max(lista), sum(lista) / len(lista)) if lista else (None, None)
        ram_pico, ram_prom = _pico_prom(self.muestras_ram_mb)
        gpu_mem_pico, gpu_mem_prom = _pico_prom(self.muestras_gpu_mem_mb)
        gpu_util_pico, gpu_util_prom = _pico_prom(self.muestras_gpu_util_pct)
        return {
            "ram_pico_mb": ram_pico, "ram_promedio_mb": ram_prom,
            "gpu_mem_pico_mb": gpu_mem_pico, "gpu_mem_promedio_mb": gpu_mem_prom,
            "gpu_util_pico_pct": gpu_util_pico, "gpu_util_promedio_pct": gpu_util_prom,
        }

print("Monitor de recursos listo (psutil:", HAY_PSUTIL, ")")


## 2.5 Cargar mejores hiperparámetros (si ya corriste `07_ajuste_hiperparametros.ipynb`)

Si existe `08_mejores_hiperparametros.json`, se usan esos valores automáticamente en
Random Forest, XGBoost y la Red Neuronal. Si no existe, se usan los valores por defecto
(sin romper nada — el pipeline corre igual aunque no hayas hecho la búsqueda todavía).


In [ ]:
import ast

def cargar_mejor_config(dir_tablas):
    ruta = os.path.join(dir_tablas, "08_mejores_hiperparametros.json")
    if not os.path.exists(ruta):
        print("No se encontro busqueda de hiperparametros (07_ajuste_hiperparametros.ipynb) -> usando valores por defecto.")
        return {}
    with open(ruta, "r", encoding="utf-8") as f:
        crudo = json.load(f)
    parseado = {modelo: ast.literal_eval(cfg) for modelo, cfg in crudo.items()}
    print("Hiperparametros cargados desde busqueda:", parseado)
    return parseado

MEJORES_HIPERPARAMETROS = cargar_mejor_config(DIR_TABLAS)
MEJOR_RF = MEJORES_HIPERPARAMETROS.get("Random Forest", {})
MEJOR_XGB = MEJORES_HIPERPARAMETROS.get("XGBoost", {})
MEJOR_NN = MEJORES_HIPERPARAMETROS.get("Red Neuronal", {})

# Salvaguarda: un batch_size chico para la NN se probo en la muestra de la busqueda,
# pero escala mal en tiempo sobre el dataset completo (5x mas lento por una ganancia
# minima de F1). Si la busqueda de 07 ya viene actualizada con tolerancia F1/tiempo,
# esto no deberia activarse nunca -- se deja como red de seguridad igual.
if MEJOR_NN.get("batch_size", 32768) < 8192:
    print(f"Aviso: batch_size encontrado en la busqueda ({MEJOR_NN.get('batch_size')}) es muy chico "
          f"para el dataset completo -> se fuerza a 32768 para que el entrenamiento no se alargue.")
    MEJOR_NN["batch_size"] = 32768


### 3.1 🌳 Random Forest (baseline)

Usa RAPIDS cuML si hay GPU disponible; si no, cae a `sklearn` automáticamente.

In [ ]:
resultados_modelos = {}  # aqui se acumulan las metricas de los 3 modelos

USA_CUML = False
try:
    import cudf, cupy as cp
    from cuml.ensemble import RandomForestClassifier as cuRF
    USA_CUML = True
except ImportError:
    from sklearn.ensemble import RandomForestClassifier as skRF

print("Backend Random Forest:", "cuML/GPU" if USA_CUML else "sklearn/CPU")

if USA_CUML:
    Xtr_rf = cudf.DataFrame.from_pandas(X_train.astype("float32"))
    ytr_rf = cudf.Series(y_train.values.astype("int32"))
    Xte_rf = cudf.DataFrame.from_pandas(X_test.astype("float32"))
    rf = cuRF(n_estimators=MEJOR_RF.get("n_estimators", 300), max_depth=MEJOR_RF.get("max_depth", 16), n_streams=4, random_state=SEMILLA)
else:
    Xtr_rf, ytr_rf, Xte_rf = X_train, y_train, X_test
    rf = skRF(n_estimators=MEJOR_RF.get("n_estimators", 300), max_depth=MEJOR_RF.get("max_depth", 16), n_jobs=-1, class_weight="balanced", random_state=SEMILLA)

t0 = time.perf_counter()
with MonitorRecursos(intervalo=0.5) as mon_rf:
    rf.fit(Xtr_rf, ytr_rf)
t_train_rf = time.perf_counter() - t0
recursos_rf = mon_rf.resumen()
ram_p = recursos_rf["ram_pico_mb"]
print(f"RAM pico: {ram_p:.0f} MB" if ram_p else "RAM: no medida")
if recursos_rf["gpu_mem_pico_mb"]:
    print(f"GPU mem pico: {recursos_rf['gpu_mem_pico_mb']:.0f} MB | GPU util promedio: {recursos_rf['gpu_util_promedio_pct']:.0f}%")

t0 = time.perf_counter()
pred_rf = rf.predict(Xte_rf)
proba_rf = rf.predict_proba(Xte_rf)
t_infer_rf = time.perf_counter() - t0

if USA_CUML:
    pred_rf = pred_rf.to_numpy() if hasattr(pred_rf, "to_numpy") else cp.asnumpy(pred_rf)
    proba_rf = proba_rf.to_numpy() if hasattr(proba_rf, "to_numpy") else cp.asnumpy(proba_rf)

joblib.dump(rf, os.path.join(DIR_MODELOS, "random_forest.pkl"))
print(f"Random Forest entrenado en {t_train_rf:.1f}s, inferencia en {t_infer_rf:.1f}s")


### 3.2 ⚡ XGBoost

In [ ]:
import xgboost as xgb

def hay_gpu():
    try:
        import subprocess
        subprocess.check_output(["nvidia-smi"])
        return True
    except Exception:
        return False

DEVICE_XGB = "cuda" if hay_gpu() else "cpu"
print("Backend XGBoost:", DEVICE_XGB)

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val, label=y_val)
dtest  = xgb.DMatrix(X_test, label=y_test)

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

params_xgb = {
    "objective": "binary:logistic",
    "eval_metric": ["auc", "logloss"],
    "tree_method": "hist",
    "device": DEVICE_XGB,
    "max_depth": MEJOR_XGB.get("max_depth", 8),
    "eta": MEJOR_XGB.get("eta", 0.1),
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "seed": SEMILLA,
}

t0 = time.perf_counter()
with MonitorRecursos(intervalo=0.5) as mon_xgb:
    xgb_modelo = xgb.train(params_xgb, dtrain, num_boost_round=500,
                            evals=[(dtrain, "train"), (dval, "val")],
                            early_stopping_rounds=25, verbose_eval=False)
t_train_xgb = time.perf_counter() - t0
recursos_xgb = mon_xgb.resumen()
ram_p = recursos_xgb["ram_pico_mb"]
print(f"RAM pico: {ram_p:.0f} MB" if ram_p else "RAM: no medida")
if recursos_xgb["gpu_mem_pico_mb"]:
    print(f"GPU mem pico: {recursos_xgb['gpu_mem_pico_mb']:.0f} MB | GPU util promedio: {recursos_xgb['gpu_util_promedio_pct']:.0f}%")

t0 = time.perf_counter()
proba_xgb = xgb_modelo.predict(dtest, iteration_range=(0, xgb_modelo.best_iteration + 1))
t_infer_xgb = time.perf_counter() - t0
pred_xgb = (proba_xgb >= 0.5).astype(int)

xgb_modelo.save_model(os.path.join(DIR_MODELOS, "xgboost.json"))
print(f"XGBoost entrenado en {t_train_xgb:.1f}s (mejor iter={xgb_modelo.best_iteration}), inferencia en {t_infer_xgb:.1f}s")


### 3.3 🧠 Red neuronal (PyTorch)

MLP totalmente conectado sobre las variables normalizadas.

In [ ]:
import os
import torch
import torch.nn as nn

DEVICE_NN = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Backend red neuronal:", DEVICE_NN)

# --- Optimizaciones para cuando se entrena en CPU (sin GPU) ------------------
if DEVICE_NN.type == "cpu":
    torch.set_num_threads(os.cpu_count())
    print(f"Usando {os.cpu_count()} hilos de CPU")
    SUBSAMPLE_FRAC_NN = min(SUBSAMPLE_FRAC_NN, 0.08)  # recorta fuerte si vas en CPU
    BATCH_SIZE_NN = MEJOR_NN.get("batch_size", 8192)
    EPOCHS = 8
else:
    BATCH_SIZE_NN = MEJOR_NN.get("batch_size", 32768)   # GPU aguanta lotes grandes -> menos iteraciones
    EPOCHS = 8

# Submuestreo opcional para entrenamiento si el dataset no cabe en memoria/GPU
if SUBSAMPLE_FRAC_NN < 1.0:
    idx = np.random.RandomState(SEMILLA).choice(len(X_train_esc), size=int(len(X_train_esc)*SUBSAMPLE_FRAC_NN), replace=False)
    X_nn_train, y_nn_train = X_train_esc[idx], y_train.values[idx]
else:
    X_nn_train, y_nn_train = X_train_esc, y_train.values

# Cast a float32 explicito (StandardScaler devuelve float64 por defecto)
X_nn_train = X_nn_train.astype(np.float32)
X_val_esc_f32 = X_val_esc.astype(np.float32)
X_test_esc_f32 = X_test_esc.astype(np.float32)

print(f"Filas de entrenamiento NN: {len(y_nn_train):,}  |  batch_size={BATCH_SIZE_NN}  |  epocas={EPOCHS}")

# Indexado vectorizado de tensores en vez de DataLoader/TensorDataset: con datos ya en
# memoria, iterar fila por fila con DataLoader es muchisimo mas lento que esto.
def iterar_batches(X, y, batch_size, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    n = X_t.shape[0]
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0, n, batch_size):
        sel = idx[i:i + batch_size]
        yield X_t[sel], y_t[sel]

class MLPIncendios(nn.Module):
    def __init__(self, n_entradas, n_clases=2):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_clases),
        )
    def forward(self, x):
        return self.red(x)

modelo_nn = MLPIncendios(n_entradas=X_train_esc.shape[1]).to(DEVICE_NN)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo_nn.parameters(), lr=MEJOR_NN.get("lr", 1e-3))

def correr_epoca_nn(X, y, batch_size, entrenar):
    modelo_nn.train(entrenar)
    perdida_total, correctos, n = 0.0, 0, 0
    for x, yb in iterar_batches(X, y, batch_size, shuffle=entrenar):
        x, yb = x.to(DEVICE_NN), yb.to(DEVICE_NN)
        if entrenar:
            optimizador.zero_grad()
        with torch.set_grad_enabled(entrenar):
            salida = modelo_nn(x)
            perdida = criterio(salida, yb)
        if entrenar:
            perdida.backward()
            optimizador.step()
        perdida_total += perdida.item() * x.size(0)
        correctos += (salida.argmax(1) == yb).sum().item()
        n += x.size(0)
    return perdida_total / n, correctos / n

historial_nn = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
t0 = time.perf_counter()
with MonitorRecursos(intervalo=0.5) as mon_nn:
    for epoca in range(1, EPOCHS + 1):
        tr_loss, tr_acc = correr_epoca_nn(X_nn_train, y_nn_train, BATCH_SIZE_NN, True)
        val_loss, val_acc = correr_epoca_nn(X_val_esc_f32, y_val.values, BATCH_SIZE_NN, False)
        historial_nn["train_loss"].append(tr_loss); historial_nn["val_loss"].append(val_loss)
        historial_nn["train_acc"].append(tr_acc);   historial_nn["val_acc"].append(val_acc)
        print(f"Epoca {epoca:>2}/{EPOCHS} | train_loss={tr_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
t_train_nn = time.perf_counter() - t0
recursos_nn = mon_nn.resumen()
ram_p = recursos_nn["ram_pico_mb"]
print(f"RAM pico: {ram_p:.0f} MB" if ram_p else "RAM: no medida")
if recursos_nn["gpu_mem_pico_mb"]:
    print(f"GPU mem pico: {recursos_nn['gpu_mem_pico_mb']:.0f} MB | GPU util promedio: {recursos_nn['gpu_util_promedio_pct']:.0f}%")

torch.save(modelo_nn.state_dict(), os.path.join(DIR_MODELOS, "red_neuronal.pt"))
print(f"\nRed neuronal entrenada en {t_train_nn:.1f}s ({DEVICE_NN})")


In [ ]:
# Inferencia sobre test para la red neuronal
modelo_nn.eval()
y_true_nn, pred_nn, proba_nn = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for x, y in iterar_batches(X_test_esc_f32, y_test.values, BATCH_SIZE_NN, shuffle=False):
        x = x.to(DEVICE_NN)
        salida = modelo_nn(x)
        probs = torch.softmax(salida, dim=1)[:, 1]
        y_true_nn.extend(y.numpy())
        pred_nn.extend(salida.argmax(1).cpu().numpy())
        proba_nn.extend(probs.cpu().numpy())
t_infer_nn = time.perf_counter() - t0
y_true_nn, pred_nn, proba_nn = np.array(y_true_nn), np.array(pred_nn), np.array(proba_nn)
print(f"Inferencia red neuronal en {t_infer_nn:.1f}s")


---
## 4. Comparar los modelos

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

def calcular_metricas(nombre, y_true, y_pred, y_score, t_train, t_infer, recursos=None):
    r = recursos or {}
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "tiempo_entrenamiento_s": t_train,
        "tiempo_inferencia_s": t_infer,
        "ram_pico_mb": r.get("ram_pico_mb"),
        "gpu_mem_pico_mb": r.get("gpu_mem_pico_mb"),
        "gpu_util_promedio_pct": r.get("gpu_util_promedio_pct"),
    }

proba_rf_pos = proba_rf[:, 1] if proba_rf.ndim == 2 else proba_rf

resultados_modelos["Random Forest"] = calcular_metricas(
    "Random Forest", y_test.values, pred_rf, proba_rf_pos, t_train_rf, t_infer_rf, recursos_rf)
resultados_modelos["XGBoost"] = calcular_metricas(
    "XGBoost", y_test.values, pred_xgb, proba_xgb, t_train_xgb, t_infer_xgb, recursos_xgb)
resultados_modelos["Red Neuronal"] = calcular_metricas(
    "Red Neuronal", y_true_nn, pred_nn, proba_nn, t_train_nn, t_infer_nn, recursos_nn)

tabla_comparativa = pd.DataFrame(resultados_modelos.values())
tabla_comparativa.to_csv(os.path.join(DIR_TABLAS, "03_comparacion_modelos.csv"), index=False)
tabla_comparativa


### 4.1 Gráfico comparativo de métricas de calidad

In [ ]:
metricas_calidad = ["accuracy", "precision", "recall", "f1", "roc_auc"]
melt = tabla_comparativa.melt(id_vars="modelo", value_vars=metricas_calidad, var_name="metrica", value_name="valor")

fig = px.bar(melt, x="metrica", y="valor", color="modelo", barmode="group",
             title="Comparacion de metricas de calidad", range_y=[0,1])
fig.show()

fig_mpl, ax = plt.subplots(figsize=(8,5))
ancho = 0.25
x_pos = np.arange(len(metricas_calidad))
for i, modelo in enumerate(tabla_comparativa["modelo"]):
    valores = tabla_comparativa[tabla_comparativa["modelo"] == modelo][metricas_calidad].values.flatten()
    ax.bar(x_pos + i*ancho, valores, width=ancho, label=modelo)
ax.set_xticks(x_pos + ancho)
ax.set_xticklabels(metricas_calidad)
ax.set_ylim(0, 1)
ax.set_title("Comparacion de metricas de calidad")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "03_comparacion_metricas.png"), dpi=150)
plt.close()


### 4.2 Curvas ROC

In [ ]:
fig_roc = go.Figure()
datos_roc = {
    "Random Forest": (y_test.values, proba_rf_pos),
    "XGBoost": (y_test.values, proba_xgb),
    "Red Neuronal": (y_true_nn, proba_nn),
}
plt.figure(figsize=(6,6))
for nombre, (yt, ys) in datos_roc.items():
    fpr, tpr, _ = roc_curve(yt, ys)
    auc = roc_auc_score(yt, ys)  # AUC exacto, con todos los puntos

    # Downsample SOLO para graficar (dataset completo puede tener millones de puntos
    # de threshold, lo que hace que Plotly se cuelgue tratando de renderizarlos)
    if len(fpr) > 2000:
        idx = np.linspace(0, len(fpr) - 1, 2000).astype(int)
        fpr_plot, tpr_plot = fpr[idx], tpr[idx]
    else:
        fpr_plot, tpr_plot = fpr, tpr

    fig_roc.add_trace(go.Scatter(x=fpr_plot, y=tpr_plot, name=f"{nombre} (AUC={auc:.3f})"))
    plt.plot(fpr_plot, tpr_plot, label=f"{nombre} (AUC={auc:.3f})")
fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash="dash", color="gray"), name="Azar"))
fig_roc.update_layout(title="Curvas ROC comparativas", xaxis_title="FPR", yaxis_title="TPR")
fig_roc.show()

plt.plot([0,1],[0,1],"--",color="gray")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Curvas ROC comparativas"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "04_curvas_roc.png"), dpi=150)
plt.close()


### 4.3 Matrices de confusión

In [ ]:
fig_cm, axes = plt.subplots(1, 3, figsize=(15,4))
predicciones = {
    "Random Forest": (y_test.values, pred_rf),
    "XGBoost": (y_test.values, pred_xgb),
    "Red Neuronal": (y_true_nn, pred_nn),
}
for ax, (nombre, (yt, yp)) in zip(axes, predicciones.items()):
    cm = confusion_matrix(yt, yp)
    ConfusionMatrixDisplay(cm, display_labels=["valida", "falsa_alarma"]).plot(ax=ax, colorbar=False)
    ax.set_title(nombre)
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "05_matrices_confusion.png"), dpi=150)
plt.show()


### 4.4 Tiempo de entrenamiento e inferencia

In [ ]:
fig_tiempo = go.Figure()
fig_tiempo.add_trace(go.Bar(x=tabla_comparativa["modelo"], y=tabla_comparativa["tiempo_entrenamiento_s"], name="Entrenamiento (s)"))
fig_tiempo.add_trace(go.Bar(x=tabla_comparativa["modelo"], y=tabla_comparativa["tiempo_inferencia_s"], name="Inferencia (s)"))
fig_tiempo.update_layout(title="Tiempo de computo por modelo", barmode="group", yaxis_title="Segundos")
fig_tiempo.show()

fig, ax = plt.subplots(figsize=(7,4))
x_pos = np.arange(len(tabla_comparativa))
ax.bar(x_pos - 0.2, tabla_comparativa["tiempo_entrenamiento_s"], width=0.4, label="Entrenamiento")
ax.bar(x_pos + 0.2, tabla_comparativa["tiempo_inferencia_s"], width=0.4, label="Inferencia")
ax.set_xticks(x_pos); ax.set_xticklabels(tabla_comparativa["modelo"])
ax.set_ylabel("Segundos"); ax.set_title("Tiempo de computo por modelo"); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "06_tiempos_computo.png"), dpi=150)
plt.close()


### 4.5 Uso de RAM y GPU durante el entrenamiento

Datos para el analisis de rendimiento de Robson (Etapa 5): pico de RAM del proceso y pico/promedio de uso de memoria y utilizacion de GPU, medidos en segundo plano durante el `fit()` de cada modelo.


In [ ]:
fig_recursos = go.Figure()
fig_recursos.add_trace(go.Bar(x=tabla_comparativa["modelo"], y=tabla_comparativa["ram_pico_mb"], name="RAM pico (MB)"))
fig_recursos.add_trace(go.Bar(x=tabla_comparativa["modelo"], y=tabla_comparativa["gpu_mem_pico_mb"], name="GPU mem pico (MB)"))
fig_recursos.update_layout(title="Uso de recursos por modelo (pico durante entrenamiento)", barmode="group", yaxis_title="MB")
fig_recursos.show()

fig, ax = plt.subplots(figsize=(7,4))
x_pos = np.arange(len(tabla_comparativa))
ax.bar(x_pos - 0.2, tabla_comparativa["ram_pico_mb"], width=0.4, label="RAM pico (MB)")
ax.bar(x_pos + 0.2, tabla_comparativa["gpu_mem_pico_mb"].fillna(0), width=0.4, label="GPU mem pico (MB)")
ax.set_xticks(x_pos); ax.set_xticklabels(tabla_comparativa["modelo"])
ax.set_ylabel("MB"); ax.set_title("Uso de recursos por modelo"); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "07_uso_recursos.png"), dpi=150)
plt.close()

print(tabla_comparativa[["modelo", "ram_pico_mb", "gpu_mem_pico_mb", "gpu_util_promedio_pct"]])


---
## 5. Resumen automático para el informe IEEE

In [ ]:
# Recalcular balance limpio justo aqui, por seguridad ante estado viejo del kernel
balance_final = (
    lf.group_by(TARGET).agg(pl.len().alias("n"))
    .sort(TARGET).collect(engine="streaming").to_pandas()
)
balance_final["pct"] = 100 * balance_final["n"].astype("float64") / balance_final["n"].sum()
print("Balance verificado justo antes del resumen:")
print(balance_final)

mejor_calidad = tabla_comparativa.sort_values("f1", ascending=False).iloc[0]
mejor_velocidad = tabla_comparativa.sort_values("tiempo_entrenamiento_s").iloc[0]

resumen = {
    "n_registros": int(n_filas),
    "n_predictoras": len(predictoras),
    "balance_clases": balance_final.to_dict(orient="records"),
    "mejor_modelo_por_f1": mejor_calidad["modelo"],
    "f1_mejor_modelo": float(mejor_calidad["f1"]),
    "roc_auc_mejor_modelo": float(mejor_calidad["roc_auc"]),
    "modelo_mas_rapido_entrenar": mejor_velocidad["modelo"],
    "tiempo_mas_rapido_s": float(mejor_velocidad["tiempo_entrenamiento_s"]),
}
with open(os.path.join(DIR_TABLAS, "04_resumen_ejecutivo.json"), "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print("RESUMEN EJECUTIVO")
print("="*50)
print(json.dumps(resumen, indent=2, ensure_ascii=False))
print("\nArchivos generados en:")
print(" -", DIR_TABLAS, "-> CSV/JSON con tablas de resultados")
print(" -", DIR_FIGURAS, "-> PNG a 150dpi listos para \\includegraphics en el informe IEEE")
print(" -", DIR_MODELOS, "-> modelos entrenados (.pkl / .json / .pt) + escalador")


### Archivos que produce este notebook

| Carpeta | Archivo | Contenido |
|---|---|---|
| `resultados/tablas/` | `01_resumen_nulos.csv` | tipos y % de nulos por columna |
| | `02_balance_clases.csv` | conteo/% por clase del target |
| | `03_comparacion_modelos.csv` | métricas de los 3 modelos (para tablas IEEE) |
| | `04_resumen_ejecutivo.json` | mejor modelo, tiempos, hallazgos clave |
| `resultados/figuras/` | `01_balance_clases.png` ... `06_tiempos_computo.png` | figuras a 150dpi listas para el informe |
| `modelos/` | `random_forest.pkl`, `xgboost.json`, `red_neuronal.pt`, `escalador.pkl` | modelos entrenados |

Con esto, la Etapa 4 queda reproducible: basta con volver a correr este notebook cuando
cambie el dataset o se ajusten hiperparámetros, y las tablas/figuras del informe se
regeneran solas.


## 6. Cómo correr la versión completa en Kabré

Este notebook con `MODO="dev"` sirve para depurar rápido en tu laptop (segundos/minutos
con una muestra del 3%). Para la corrida real con el dataset completo (~52M filas), usa
Kabré y solo cambia `MODO = "full"`:

```bash
# 1) Subir la carpeta modelado/ (P mayuscula en scp)
scp -P 22022 -r modelado ulead-17@kabre.cenat.ac.cr:~/

# 2) El dataset grande va en /data, NO en /home (cuota de 10GB)
ssh -p 22022 ulead-17@kabre.cenat.ac.cr
mkdir -p /data/$USER
# copiar/descargar dataset_modelo.parquet ahi; ajustar RUTA_DATASET si aplica

# 3a) Opcion JupyterHub (mas simple): abrir este .ipynb en JupyterHub de Kabre,
#     cambiar MODO="full" y correr todas las celdas con GPU asignada.

# 3b) Opcion batch (como el benchmark de Esteban):
jupyter nbconvert --to script 00_pipeline_modelado.ipynb
# editar la copia .py: MODO = "full"
sbatch submit_kabre.slurm   # reutilizando/adaptando el que ya armo Esteban
squeue -u $USER
tail -f bench_<jobid>.out
```

**Importante:** antes de la entrega final, todos los resultados/figuras/modelos que vayan
al informe IEEE deben salir de una corrida con `MODO="full"` en Kabré, no de la muestra
de desarrollo local.
